[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ruben-tsui/TTBook2026/blob/main/OpusTools.ipynb)

### STEP (1) Set up environment and libraries

In [ ]:
!pip install -q opustools langcodes language_data
import subprocess
import regex as re
import ipywidgets as widgets
from IPython.display import display
from langcodes import Language

### STEP (2) Search OPUS for corpus

In [ ]:
Source_lang = 'en' # @param ['en', 'zh_TW', 'zh_tw', 'zh_CN', 'zh_cn', 'ja', 'es', 'es_ES', 'fr']
Target_lang = 'zh_tw' # @param ['en', 'zh_TW', 'zh_tw', 'zh_CN', 'zh_cn', 'ja', 'es', 'es_ES', 'fr']
Corpus = 'TED2020' # @param {"type":"string"}

#!opus_get -s "$Source_lang" -t "$Target_lang" -p tmx -l|egrep -i "$Corpus"
#!opus_get -s "$Source_lang" -t "$Target_lang" -p tmx -l|egrep -i "$Corpus"


# 1. Fetch available files from opus_get
cmd = f'opus_get -s "{Source_lang}" -t "{Target_lang}" -p tmx -l'
try:
    result = subprocess.check_output(cmd, shell=True).decode('utf-8')
except subprocess.CalledProcessError:
    result = ""

# 2. Parse lines to get (url, full_info)
options = []
url_map = {}

lines = result.strip().split('\n')
for line in lines:
    if Corpus.lower() in line.lower() and 'http' in line:
        url_match = re.search(r'https?://\S+', line)
        if url_match:
            url = url_match.group(0)
            display_text = line.strip()
            options.append(display_text)
            url_map[display_text] = url

if not options:
    print(f"No matching versions for '{Corpus}' found. Try a broader search.")
else:
    print(f"Found {len(options)} version(s) for {Corpus}:")
    for opt in options:
        print(f"- {opt}")

# @markdown Select a specific version from the search results (or leave blank to pick the first one):
#Selected_Version = "" # @param {type:"string"}

#

### STEP (3) Download Corpus
Run the cell below to parse the search results and select which version to download.

In [ ]:
# 1. Fetch available files from opus_get
cmd = f'opus_get -s "{Source_lang}" -t "{Target_lang}" -p tmx -l'
try:
    result = subprocess.check_output(cmd, shell=True).decode('utf-8')
except subprocess.CalledProcessError:
    result = ""

# 2. Parse results
options = []
url_map = {}
lines = result.strip().split('\n')
for line in lines:
    if Corpus.lower() in line.lower() and 'http' in line:
        url_match = re.search(r'https?://\S+', line)
        if url_match:
            url = url_match.group(0)
            display_text = line.strip()
            options.append(display_text)
            url_map[display_text] = url

if not options:
    print(f"No matching versions for '{Corpus}' found.")
else:
    print(f"Found {len(options)} version(s). Please select one from the dropdown:")
    # Create a dynamic dropdown
    version_dropdown = widgets.Dropdown(
        options=options,
        description='Version:',
        disabled=False,
        layout={'width': 'max-content'}
    )
    display(version_dropdown)

    # Function to update the global variable used by the download cell
    def on_change(change):
        global Selected_Version
        if change['type'] == 'change' and change['name'] == 'value':
            Selected_Version = change['new']

    version_dropdown.observe(on_change)
    # Initialize with the first option
    Selected_Version = options[0]

#### main logic

In [ ]:
# @title Download Selected Corpus
import os

# Find the URL for the selected version
# If 'Selected_Version' is empty, we'll take the first one found as a default
chosen_url = ""
for opt in options:
    if Selected_Version in opt and Selected_Version != "":
        chosen_url = url_map[opt]
        break
if not chosen_url and options:
    chosen_url = url_map[options[0]]

if chosen_url:
    # Standardize name: Corpus.src-trg.tmx.gz
    extension = ".tmx.gz" if chosen_url.endswith(".gz") else ".tmx"
    standard_name = f"{Corpus}.{Source_lang}-{Target_lang}{extension}"

    print(f"Downloading: {chosen_url}")
    print(f"Saving as: {standard_name}")

    !wget -O "{standard_name}" "{chosen_url}"
    tmx_path = os.path.abspath(standard_name)
else:
    print("No URL found to download.")

In [ ]:
from IPython.display import Markdown, display, HTML
import regex as re

RED     = "\033[31m"
GREEN   = "\033[32m"
YELLOW  = "\033[33m"
BLUE    = "\033[34m"
MAGENTA = "\033[35m"
CYAN    = "\033[36m"
WHITE   = "\033[37m"
RESET   = "\033[0m"  # Resets to default color

def addcolor(re_target, original_string, insert_str1="{RED}", insert_str2="{RESET}"):

    matches = re_target.finditer(original_string)
    index_pairs = [(m.start(), m.end()) for m in matches]

    insertions = []
    for start, end in index_pairs:
        insertions.append((start, insert_str1))
        insertions.append((end, insert_str2))

    # Sort insertions by index in reverse order to avoid index shifting
    insertions.sort(reverse=True)

    # Convert string to list for easier manipulation
    result = list(original_string)

    # Insert strings at each index from right to left
    for index, string in insertions:
        result.insert(index, string)

    # Join the list back into a string
    return ''.join(result)


In [ ]:
#search = r'click'
# Reproduce the exact results without blank lines and without color codes
#!rg -z --color never -A1 "$search" "$standard_name" | rg --color never -o '<seg>(.*?)</seg>' -r '$1'

In [ ]:
import gzip
import xml.etree.ElementTree as ET
import regex as re
from IPython.display import HTML, display

def stream_tmx_search(file_path, search_pattern, source_lang_code, target_lang_code, limit=100000):
    """Streams a TMX file and yields matches for the search_pattern."""
    re_search = re.compile(search_pattern, re.IGNORECASE)
    count = 0

    with gzip.open(file_path, 'rb') as f:
        # Using iterparse for memory efficiency
        context = ET.iterparse(f, events=('end',))
        for event, elem in context:
            if elem.tag == 'tu':
                pair = {}
                for tuv in elem.findall('tuv'):
                    lang = tuv.get('{http://www.w3.org/XML/1998/namespace}lang')
                    seg_node = tuv.find('seg')
                    # Ensure seg is a string even if node is empty
                    seg = str(seg_node.text) if (seg_node is not None and seg_node.text is not None) else ""
                    if lang == source_lang_code:
                        pair[source_lang_code] = seg
                    elif lang == target_lang_code:
                        pair[target_lang_code] = seg

                if source_lang_code in pair and target_lang_code in pair:
                    # Check for match before yielding
                    if re_search.search(pair[source_lang_code]) or re_search.search(pair[target_lang_code]):
                        yield count, pair[source_lang_code], pair[target_lang_code]

                count += 1
                elem.clear() # Discard the element from memory
                if count >= limit: break

In [ ]:
standard_name

## STEP (4) Search

In [ ]:
# Re-executing the memory-efficient search with the fix applied
tmx_path = standard_name # param {"type":"string"}
search_term = 'enhance.+(resilience|capabilities|transparency|competitiveness|efficiency)' # @param {"type":"string"}
re_search = re.compile(search_term, re.IGNORECASE)

rows = ""
# Stream up to 2,000,000 pairs to ensure we find matches
cnt = 0
for original_idx, source_segment, target_segment in stream_tmx_search(tmx_path, search_term, Source_lang, Target_lang, limit=2000000):
    cnt += 1
    source_html = f"<span style='font-size: 16px; font-family: Consolas'>{source_segment}</span>"
    target_html = f"<span style='font-size: 16px; font-family: Microsoft Jhenghei'>{target_segment}</span>"

    # Use the addcolor function to highlight matches
    source_html = addcolor(re_search, source_html).replace('{RED}', '<span style="color:red; font-weight:bold;">').replace('{RESET}', '</span>')
    target_html = addcolor(re_search, target_html).replace('{RED}', '<span style="color:red; font-weight:bold;">').replace('{RESET}', '</span>')

    rows += f"""
    <tr>
    <td style='padding: 5px;' width='8'>{original_idx}</td>
    <td style='padding: 5px;' width='46%'>{source_html}</td>
    <td style='padding: 5px;' width='46%'>{target_html}</td>
    </tr>
    """

Source_lang_fullname = Language.get(Source_lang).display_name('en')
Target_lang_fullname = Language.get(Target_lang).display_name('en')

html_output = f"""
<table border="1" style="border-collapse: collapse; width: 100%; text-align: left;">
<tr style="background-color: #CCCCCC;"><th>Original Index</th><th style="text-align: center;">{Source_lang_fullname}</th><th style="text-align: center;">{Target_lang_fullname}</th></tr>
{rows}
</table>
Total: {cnt} results
"""
display(HTML(html_output))